# Pneumonia Detection from Chest X-Rays - DenseNet121

Binary classifier: NORMAL vs PNEUMONIA on the Kaggle Chest X-Ray Pneumonia dataset
(pediatric AP chest x-rays, 5,863 images, Kermany et al. Cell 2018).

**Architecture:** DenseNet121 (ImageNet pretrained) -> GAP -> dropout 0.3 -> single logit (BCE).
**Protocol:** two-phase fine-tune (frozen head warmup -> full unfreeze), recall-first threshold
tuning on a carved validation split, single locked evaluation on the held-out 624-image test set.

Dataset caveats (stated upfront):
- native val split is 16 images and statistically useless; we carve a stratified 10% val from train instead
- labels are physician-graded but noisy; Grad-CAM review of FPs doubles as a label-noise audit
- pediatric single-center data; demo-grade portfolio piece, not a diagnostic device

In [ ]:
# ---- config ----
QUICK = False          # True = tiny smoke run to validate the pipeline end to end
SEED = 42
IMG_SIZE = 224
BATCH = 32
P1_EPOCHS = 5          # phase 1: head only, backbone frozen
P2_EPOCHS = 15         # phase 2: full unfreeze
P1_LR = 1e-3
P2_LR = 1e-4
WEIGHT_DECAY = 1e-4
ETA_MIN = 1e-6
LABEL_SMOOTH = 0.05
DROP_RATE = 0.3
PATIENCE = 4           # early stop on val AUROC
TARGET_RECALL = 0.95   # recall-first operating point
N_BOOT = 1000

if QUICK:
    P1_EPOCHS, P2_EPOCHS, N_BOOT = 1, 1, 100

import os, glob, json, time, random
import numpy as np
import torch
import torch.nn as nn
import timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
                             precision_recall_curve, roc_curve)
import matplotlib.pyplot as plt

def seed_everything(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else '')
print('torch', torch.__version__, '| timm', timm.__version__)

In [ ]:
# ---- data: locate dataset, build splits ----
roots = glob.glob('/kaggle/input/*/chest_xray') + glob.glob('/kaggle/input/*/*/*/chest_xray')
assert roots, 'dataset not found - attach paultimothymooney/chest-xray-pneumonia as notebook input'
ROOT = roots[0]
print('dataset root:', ROOT)

def collect(split):
    items = []
    for label, cls in [(0, 'NORMAL'), (1, 'PNEUMONIA')]:
        for p in sorted(glob.glob(os.path.join(ROOT, split, cls, '*.jpeg'))):
            items.append((p, label))
    return items

train_pool = collect('train')   # 5,216 (1,341 normal / 3,875 pneumonia)
test_items = collect('test')    # 624 (234 normal / 390 pneumonia) - LOCKED until final eval
print(f'train pool: {len(train_pool)} | test (locked): {len(test_items)}')

paths = [p for p, _ in train_pool]
labels = np.array([y for _, y in train_pool])
tr_idx, va_idx = train_test_split(np.arange(len(paths)), test_size=0.10,
                                  stratify=labels, random_state=SEED)
train_items = [train_pool[i] for i in tr_idx]
val_items   = [train_pool[i] for i in va_idx]

if QUICK:
    train_items, val_items, test_items = train_items[:400], val_items[:100], test_items[:128]

n_norm = sum(1 for _, y in train_items if y == 0)
n_pneu = sum(1 for _, y in train_items if y == 1)
POS_WEIGHT = n_norm / n_pneu
print(f'train: {len(train_items)} ({n_norm} normal / {n_pneu} pneumonia) | val: {len(val_items)}')
print(f'pos_weight = {POS_WEIGHT:.3f}')

In [ ]:
# ---- datasets + loaders ----
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.RandomAffine(degrees=0, translate=(0.10, 0.10), scale=(0.90, 1.10)),
    T.ColorJitter(brightness=0.10, contrast=0.10),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class XRayDS(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        img = Image.open(p).convert('RGB')
        return self.tf(img), float(y), p

def make_loader(items, tf, shuffle):
    return DataLoader(XRayDS(items, tf), batch_size=BATCH, shuffle=shuffle,
                      num_workers=2, pin_memory=True, drop_last=shuffle)

train_dl = make_loader(train_items, train_tf, True)
val_dl   = make_loader(val_items, eval_tf, False)
test_dl  = make_loader(test_items, eval_tf, False)
len(train_dl), len(val_dl), len(test_dl)

In [ ]:
# ---- model: DenseNet121, single logit ----
model = timm.create_model('densenet121', pretrained=True, num_classes=1, drop_rate=DROP_RATE)
model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'DenseNet121 params: {n_params/1e6:.1f}M | head: GAP -> dropout {DROP_RATE} -> 1 logit')

In [ ]:
# ---- train/eval loops ----
def smooth_targets(y, ls=LABEL_SMOOTH):
    return y * (1 - ls) + 0.5 * ls

def run_epoch(dl, train=True, optimizer=None, pos_weight=None):
    model.train() if train else model.eval()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    total_loss, probs_all, ys_all, n = 0.0, [], [], 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y, _ in dl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type=='cuda'):
                logit = model(x).squeeze(1)
                loss = loss_fn(logit, smooth_targets(y)) if train else loss_fn(logit, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0); n += x.size(0)
            probs_all.append(torch.sigmoid(logit.detach().float()).cpu().numpy())
            ys_all.append(y.detach().cpu().numpy())
    probs = np.concatenate(probs_all); ys = np.concatenate(ys_all)
    auroc = roc_auc_score(ys, probs) if len(np.unique(ys)) > 1 else float('nan')
    return total_loss / n, auroc, probs, ys

def predict(dl):
    model.eval(); probs_all, ys_all, paths_all = [], [], []
    with torch.no_grad():
        for x, y, p in dl:
            x = x.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type=='cuda'):
                logit = model(x).squeeze(1)
            probs_all.append(torch.sigmoid(logit.float()).cpu().numpy())
            ys_all.append(y.numpy()); paths_all += list(p)
    return np.concatenate(probs_all), np.concatenate(ys_all), paths_all

history = []
best_auroc, best_state, bad_epochs = -1.0, None, 0
t0 = time.time()

In [ ]:
# ---- phase 1: head only, backbone frozen ----
for n, p in model.named_parameters():
    p.requires_grad = 'classifier' in n
head_params = [p for p in model.parameters() if p.requires_grad]
print(f'phase 1: {sum(p.numel() for p in head_params):,} trainable params (head only)')
opt = torch.optim.Adam(head_params, lr=P1_LR)
pw = torch.tensor(POS_WEIGHT, device=DEVICE)

for ep in range(1, P1_EPOCHS + 1):
    tr_loss, tr_auc, _, _ = run_epoch(train_dl, True, opt, pw)
    va_loss, va_auc, _, _ = run_epoch(val_dl, False)
    history.append(dict(phase=1, epoch=ep, train_loss=tr_loss, val_loss=va_loss, val_auroc=va_auc))
    print(f'[P1 {ep}/{P1_EPOCHS}] train_loss {tr_loss:.4f} | val_loss {va_loss:.4f} | val_AUROC {va_auc:.4f}')
    if va_auc > best_auroc:
        best_auroc, bad_epochs = va_auc, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad_epochs += 1

In [ ]:
# ---- phase 2: full unfreeze, cosine decay, early stop on val AUROC ----
for p in model.parameters():
    p.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=P2_LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=P2_EPOCHS, eta_min=ETA_MIN)
print(f'phase 2: all {sum(p.numel() for p in model.parameters()):,} params unfrozen')

for ep in range(1, P2_EPOCHS + 1):
    tr_loss, tr_auc, _, _ = run_epoch(train_dl, True, opt, pw)
    va_loss, va_auc, _, _ = run_epoch(val_dl, False)
    sched.step()
    history.append(dict(phase=2, epoch=ep, train_loss=tr_loss, val_loss=va_loss, val_auroc=va_auc))
    print(f'[P2 {ep}/{P2_EPOCHS}] train_loss {tr_loss:.4f} | val_loss {va_loss:.4f} | val_AUROC {va_auc:.4f} | lr {sched.get_last_lr()[0]:.2e}')
    if va_auc > best_auroc:
        best_auroc, bad_epochs = va_auc, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f'early stop: no val AUROC gain for {PATIENCE} epochs')
            break

print(f'\ntotal wall: {(time.time()-t0)/60:.1f} min | best val AUROC {best_auroc:.4f}')
model.load_state_dict(best_state)
torch.save({'state_dict': best_state, 'val_auroc': best_auroc, 'config': dict(
    arch='densenet121', img_size=IMG_SIZE, pos_weight=POS_WEIGHT,
    p1=dict(epochs=P1_EPOCHS, lr=P1_LR), p2=dict(epochs=P2_EPOCHS, lr=P2_LR, wd=WEIGHT_DECAY),
    label_smooth=LABEL_SMOOTH, drop_rate=DROP_RATE)},
    '/kaggle/working/densenet121_pneumonia_best.pth')
print('saved /kaggle/working/densenet121_pneumonia_best.pth')

In [ ]:
# ---- training curves ----
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
xs = range(1, len(history) + 1)
ax[0].plot(xs, [h['train_loss'] for h in history], label='train')
ax[0].plot(xs, [h['val_loss'] for h in history], label='val')
ax[0].set_title('loss'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(xs, [h['val_auroc'] for h in history], color='darkorange')
ax[1].set_title('val AUROC'); ax[1].set_xlabel('epoch'); ax[1].grid(alpha=.3)
ax[1].axvline(P1_EPOCHS + .5, ls='--', c='gray', alpha=.6)
plt.tight_layout(); plt.savefig('/kaggle/working/training_curves.png', dpi=110); plt.show()

In [ ]:
# ---- recall-first: threshold selection on VAL ----
va_probs, va_ys, _ = predict(val_dl)
prec, rec, thr = precision_recall_curve(va_ys, va_probs)
thr = np.append(thr, 1.0)

# primary: smallest threshold achieving recall >= TARGET_RECALL with max precision
ok = np.where(rec >= TARGET_RECALL)[0]
if len(ok):
    best_i = ok[np.argmax(prec[ok])]
else:
    best_i = int(np.argmax(rec))
    print('WARNING: target recall unreachable on val, using max-recall point')
TH_RECALL = float(thr[best_i])

# secondary: max F2
f2 = (5 * prec * rec) / (4 * prec + rec + 1e-9)
TH_F2 = float(thr[int(np.argmax(f2))])

print(f'val AUROC {roc_auc_score(va_ys, va_probs):.4f} | val AUPRC {average_precision_score(va_ys, va_probs):.4f}')
print(f'TH_RECALL (recall>={TARGET_RECALL}): {TH_RECALL:.4f} -> val recall {rec[best_i]:.3f}, precision {prec[best_i]:.3f}')
print(f'TH_F2 (max F2): {TH_F2:.4f}')

In [ ]:
# ---- LOCKED TEST EVAL (624 images, run once) ----
te_probs, te_ys, te_paths = predict(test_dl)

def metrics_at(y, p, t):
    yhat = (p >= t).astype(int)
    cm = confusion_matrix(y, yhat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return dict(accuracy=(tp+tn)/len(y), precision=tp/max(tp+fp,1), recall=tp/max(tp+fn,1),
                specificity=tn/max(tn+fp,1),
                f1=2*tp/max(2*tp+fp+fn,1), f2=5*tp/max(5*tp+4*fn+fp,1),
                roc_auc=roc_auc_score(y, p), pr_auc=average_precision_score(y, p),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def bootstrap(y, p, t, n=N_BOOT, seed=0):
    rng = np.random.default_rng(seed); keys = ['accuracy','precision','recall','specificity','f1','roc_auc']
    vals = {k: [] for k in keys}
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2: continue
        m = metrics_at(y[idx], p[idx], t)
        for k in keys: vals[k].append(m[k])
    return {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) for k, v in vals.items()}

res_recall = metrics_at(te_ys, te_probs, TH_RECALL)
res_f2     = metrics_at(te_ys, te_probs, TH_F2)
res_050    = metrics_at(te_ys, te_probs, 0.5)
ci = bootstrap(te_ys, te_probs, TH_RECALL)

summary = dict(operating_point=dict(threshold=round(TH_RECALL,4), target_recall=TARGET_RECALL, **{k: round(v,4) for k,v in res_recall.items()}),
               max_f2_point=dict(threshold=round(TH_F2,4), **{k: round(v,4) for k,v in res_f2.items()}),
               default_0p5=dict(**{k: round(v,4) for k,v in res_050.items()}),
               bootstrap_95ci={k: [round(a,4), round(b,4)] for k,(a,b) in ci.items()},
               n_test=len(te_ys), n_normal=int((te_ys==0).sum()), n_pneumonia=int((te_ys==1).sum()))
print(json.dumps(summary, indent=2))
with open('/kaggle/working/metrics.json','w') as f: json.dump(summary, f, indent=2)

In [ ]:
# ---- ROC + PR curves, confusion matrix ----
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
fpr, tpr, _ = roc_curve(te_ys, te_probs)
ax[0].plot(fpr, tpr); ax[0].plot([0,1],[0,1],'--',c='gray')
ax[0].set_title(f"ROC (AUC {res_recall['roc_auc']:.3f})"); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].grid(alpha=.3)
p2, r2, _ = precision_recall_curve(te_ys, te_probs)
ax[1].plot(r2, p2); ax[1].set_title(f"PR (AUPRC {res_recall['pr_auc']:.3f})")
ax[1].set_xlabel('recall'); ax[1].set_ylabel('precision'); ax[1].grid(alpha=.3)
cm = confusion_matrix(te_ys, (te_probs >= TH_RECALL).astype(int), labels=[0,1])
im = ax[2].imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax[2].text(j, i, cm[i,j], ha='center', va='center', fontsize=16)
ax[2].set_xticks([0,1]); ax[2].set_yticks([0,1])
ax[2].set_xticklabels(['pred NORMAL','pred PNEUMONIA']); ax[2].set_yticklabels(['true NORMAL','true PNEUMONIA'])
ax[2].set_title(f'confusion matrix @ thr {TH_RECALL:.3f}')
plt.tight_layout(); plt.savefig('/kaggle/working/roc_pr_cm.png', dpi=110); plt.show()

In [ ]:
# ---- Grad-CAM galleries: TP / TN / FP / FN at the recall-first operating point ----
class GradCAM:
    def __init__(self, m, layer):
        self.m, self.act, self.grad = m, None, None
        layer.register_forward_hook(lambda _m,_i,o: setattr(self,'act',o.detach()))
        layer.register_full_backward_hook(lambda _m,_gi,go: setattr(self,'grad',go[0].detach()))
    def __call__(self, x):
        self.m.zero_grad(set_to_none=True)
        logit = self.m(x).squeeze(1)
        logit.backward(torch.ones_like(logit))
        w = self.grad.mean(dim=(2,3), keepdim=True)
        cam = torch.relu((w * self.act).sum(1)).squeeze(0).cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-9)
        return cam

gcam = GradCAM(model, model.features.denseblock4.denselayer16.conv2)
yhat = (te_probs >= TH_RECALL).astype(int)
rng = np.random.RandomState(0)
def rand_pick(mask, n=4):
    ix = np.where(mask)[0]; return list(rng.choice(ix, min(n, len(ix)), replace=False))
def worst_pick(mask, n=4, high=True):
    ix = np.where(mask)[0]; ix = ix[np.argsort(-te_probs[ix] if high else te_probs[ix])]; return list(ix[:n])
m_tp=(yhat==1)&(te_ys==1); m_tn=(yhat==0)&(te_ys==0); m_fp=(yhat==1)&(te_ys==0); m_fn=(yhat==0)&(te_ys==1)
print(f'TP {m_tp.sum()} | TN {m_tn.sum()} | FP {m_fp.sum()} | FN {m_fn.sum()}')
NAME = {0:'NORMAL', 1:'PNEUMONIA'}
groups = [('TRUE POSITIVE (pneumonia caught, random)', rand_pick(m_tp)),
          ('TRUE NEGATIVE (healthy cleared, random)', rand_pick(m_tn)),
          ('FALSE POSITIVE (false alarm, most confident)', worst_pick(m_fp, high=True)),
          ('FALSE NEGATIVE (missed pneumonia, most confident miss)', worst_pick(m_fn, high=False))]
def draw(idxs, title, path):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.6))
    for c in range(4):
        ax = axes[c]; ax.axis('off')
        if c >= len(idxs): continue
        i = idxs[c]
        img = Image.open(te_paths[i]).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        x, _, _ = XRayDS([test_items[i]], eval_tf)[0]
        cam = gcam(x.unsqueeze(0).to(DEVICE))
        cam_img = Image.fromarray(np.uint8(255*cam)).resize((IMG_SIZE, IMG_SIZE))
        ax.imshow(img); ax.imshow(cam_img, cmap='jet', alpha=0.35)
        ttl = 'actual ' + NAME[int(te_ys[i])] + chr(10) + 'pred ' + NAME[int(yhat[i])] + '  p(pneu)=' + f'{te_probs[i]:.2f}'
        ax.set_title(ttl, fontsize=10)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(path, dpi=110); plt.show()
fnames = ['gradcam_TP.png','gradcam_TN.png','gradcam_FP.png','gradcam_FN.png']
for (t, ix), f in zip(groups, fnames):
    draw(ix, t, '/kaggle/working/'+f)
# combined grid
fig, axes = plt.subplots(4, 4, figsize=(16, 18))
for r, (t, idxs) in enumerate(groups):
    for c in range(4):
        ax = axes[r, c]; ax.axis('off')
        if c >= len(idxs): continue
        i = idxs[c]
        img = Image.open(te_paths[i]).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        x, _, _ = XRayDS([test_items[i]], eval_tf)[0]
        cam = gcam(x.unsqueeze(0).to(DEVICE))
        ax.imshow(img); ax.imshow(Image.fromarray(np.uint8(255*cam)).resize((IMG_SIZE, IMG_SIZE)), cmap='jet', alpha=0.35)
        lbl = (t.split(' (')[0] + chr(10) if c == 0 else '') + 'actual ' + NAME[int(te_ys[i])] + ' / pred ' + NAME[int(yhat[i])] + '  p=' + f'{te_probs[i]:.2f}'
        ax.set_title(lbl, fontsize=9)
plt.tight_layout(); plt.savefig('/kaggle/working/gradcam_grid.png', dpi=110); plt.show()
# per-sample predictions for the whole test set
import csv
with open('/kaggle/working/test_predictions.csv','w',newline='') as fh:
    w = csv.writer(fh); w.writerow(['path','actual','pred','p_pneumonia','outcome'])
    for i in range(len(te_ys)):
        o = 'TP' if m_tp[i] else 'TN' if m_tn[i] else 'FP' if m_fp[i] else 'FN'
        w.writerow([te_paths[i], int(te_ys[i]), int(yhat[i]), f'{te_probs[i]:.4f}', o])
print('Grad-CAM off denseblock4 last conv. FP/FN = most confident errors (worst cases), TP/TN random seed 0.')

In [ ]:
# ---- outputs ----
print(os.listdir('/kaggle/working'))